# P4 final matched-resistance $\delta_{\mathrm{SEI}}$ / $\delta_{\mathrm{plating}}$ ratios

This cleaned notebook keeps the optimized first life and optimized second-life duty cycle fixed. At the second-life restart, only the split between $\delta_{\mathrm{SEI}}$ and $\delta_{\mathrm{plating}}$ changes while the combined film area-specific resistance remains identical.

The three splits are 10/90, 50/50, and 90/10 for SEI/plating film resistance. The notebook also includes the C/20 charge-discharge reference tests at the beginning, middle, and end of each second-life trajectory.

## Load data and validate matched resistance

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Locate the deliverable folder from the working directory, so the notebook
# runs from a fresh checkout with no machine-specific path.
FINAL_ROOT = Path.cwd()
if not (FINAL_ROOT / "data").exists():
    for _base in (Path.cwd(), *Path.cwd().parents):
        _cand = _base / "deepSOH_final_august22_2026"
        if (_cand / "data").exists():
            FINAL_ROOT = _cand
            break
DATA_ROOT = FINAL_ROOT / "data" / "delta_ratio"
OUTPUT_ROOT = FINAL_ROOT / "outputs" / "delta_ratio"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

first = pd.read_csv(DATA_ROOT / "P4_optimized_first_life_trajectory.csv")
second = pd.read_csv(DATA_ROOT / "P4_three_matched_second_life_trajectories.csv")
states = pd.read_csv(DATA_ROOT / "P4_matched_initial_states.csv")
rpt = pd.read_csv(DATA_ROOT / "P4_three_matched_second_life_C20_RPTs.csv")
summary = json.loads((DATA_ROOT / "P4_three_matched_second_life_summary.json").read_text())

LABELS = {
    "low_SEI_high_plating": r"10% $\delta_{\mathrm{SEI}}$ / 90% $\delta_{\mathrm{plating}}$",
    "moderate_SEI_moderate_plating": r"26% $\delta_{\mathrm{SEI}}$ / 74% $\delta_{\mathrm{plating}}$",
    "high_SEI_low_plating": r"90% $\delta_{\mathrm{SEI}}$ / 10% $\delta_{\mathrm{plating}}$",
}
for case, label in LABELS.items():
    states.loc[states["case"] == case, "label"] = label
    second.loc[second["case"] == case, "label"] = label
    rpt.loc[rpt["case"] == case, "label"] = label

np.testing.assert_allclose(
    states["constructed_film_ASR_Ohm_m2"], states["target_film_ASR_Ohm_m2"],
    rtol=1e-12, atol=1e-15,
)
np.testing.assert_allclose(
    states[["nLi_mol", "Cp_Ah", "Cn_Ah"]].to_numpy(),
    np.repeat(
        states[["nLi_mol", "Cp_Ah", "Cn_Ah"]]
        .iloc[0]
        .to_numpy()[None, :],
        len(states),
        axis=0,
    ),
    rtol=1e-12, atol=1e-15,
)
display(states[[
    "label", "alpha_SEI_film_ASR", "alpha_plating_film_ASR",
    "delta_SEI_m", "delta_pl_m", "constructed_film_ASR_Ohm_m2",
    "initial_capacity_Ah", "initial_resistance_Ohm", "safe_last_cycle",
    "safe_final_retention", "maximum_scaled_initial_error",
]])

## Matched-resistance second lives (second use only)

In [ ]:
def plot_life(output_path):
    panels = (
        ("capacity_Ah", "Capacity [Ah]"),
        ("resistance_Ohm", "Resistance [Ohm]"),
        ("expansion_um", "Irreversible expansion [um]"),
    )
    fig, axes = plt.subplots(3, 1, figsize=(11, 11), sharex=True, constrained_layout=True)
    for _, state in states.iterrows():
        block = second.loc[second["case"] == state["case"]].sort_values("cycle")
        for index, (column, ylabel) in enumerate(panels):
            axes[index].plot(
                block["cycle"], block[column], color=state["color"], linewidth=1.9,
                label=state["label"] if index == 0 else None,
            )
            axes[index].set_ylabel(ylabel)
            axes[index].grid(True, alpha=0.28)
    axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, 1.36), ncol=2, fontsize=8.2)
    axes[-1].set_xlabel("Cycle number in second life")
    fig.suptitle("P4 matched-resistance second lives", fontsize=13)
    fig.savefig(output_path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()

plot_life(OUTPUT_ROOT / "P4_delta_ratio_second_life.png")

## C/20 charge-discharge reference tests

In [ ]:
stage_order = ("beginning", "middle", "end")
fig, axes = plt.subplots(3, 1, figsize=(9, 12), sharex=True, constrained_layout=True)
for row, stage in enumerate(stage_order):
    for _, state in states.iterrows():
        block = rpt.loc[
            (rpt["case"] == state["case"]) & (rpt["stage"] == stage)
        ].sort_values("time_h")
        time = block["time_h"] - block["time_h"].iloc[0]
        axes[row].plot(time, block["voltage_V"], color=state["color"],
                       linewidth=1.7, label=state["label"])
    axes[row].set_title(f"{stage.title()} of second life")
    axes[row].set_ylabel("Voltage [V]")
    axes[row].grid(True, alpha=0.25)
axes[-1].set_xlabel("RPT time [h]")
axes[0].legend(loc="best", fontsize=8)
fig.suptitle("P4 C/20 beginning, middle, and end reference tests", fontsize=14)
rpt_png = OUTPUT_ROOT / "P4_delta_ratio_C20_RPTs.png"
fig.savefig(rpt_png, dpi=220, bbox_inches="tight", facecolor="white")
plt.show()

rpt_summary = rpt.groupby(["case", "label", "stage"], as_index=False).agg(
    actual_cycle=("actual_cycle", "first"),
    duration_h=("time_h", "max"),
    minimum_voltage_V=("voltage_V", "min"),
    maximum_voltage_V=("voltage_V", "max"),
)
display(rpt_summary)
rpt_summary.to_csv(OUTPUT_ROOT / "P4_delta_ratio_C20_summary.csv", index=False)

## Export plotting data to MATLAB (.mat)

In [ ]:
from scipy.io import savemat

MAT_ROOT = FINAL_ROOT / "matlab"
MAT_ROOT.mkdir(parents=True, exist_ok=True)

def _hex2rgb(h):
    h = h.lstrip("#")
    return [int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4)]

_cols = ["cycle", "capacity_Ah", "resistance_Ohm", "expansion_um"]
_stage_key = {"beginning": "beginning", "middle": "middle", "end": "ending"}
mat = {}
_cases, _colors = [], []
for _, st in states.iterrows():
    case = st["case"]
    _cases.append(case)
    block = second[second["case"] == case].sort_values("cycle")
    d = {c: block[c].to_numpy(dtype=float) for c in _cols}
    rpt_d = {}
    for stage, key in _stage_key.items():
        blk = rpt[(rpt["case"] == case) & (rpt["stage"] == stage)].sort_values("time_h")
        t = blk["time_h"].to_numpy(dtype=float)
        t = (t - t[0]) if len(t) else t
        rpt_d[key] = {"time_h": t, "voltage_V": blk["voltage_V"].to_numpy(dtype=float),
                   "cycle": (int(blk["actual_cycle"].iloc[0]) if len(blk) else -1)}
    d["rpt"] = rpt_d
    mat[case] = d
    _colors.append(_hex2rgb(st["color"]))
mat["cases"] = np.array(_cases, dtype=object)
mat["colors"] = np.array(_colors, dtype=float)
savemat(str(MAT_ROOT / "P4_03_delta_ratio.mat"), mat, do_compression=True)
print("saved", MAT_ROOT / "P4_03_delta_ratio.mat")